# SFINCS — New Jersey Coastal Flood Model (Hurricane Sandy)

A **step-by-step, build-from-the-ground-up** guide to a compound coastal-flood
simulation with [SFINCS](https://sfincs.readthedocs.io) + [hydromt-sfincs](https://deltares.github.io/hydromt_sfincs/),
hindcasting **Hurricane Sandy (29 Oct 2012)** on the northern NJ coast
(Sandy Hook → Asbury Park).

The notebook is deliberately incremental: every major step (grid, elevation,
mask, subgrid, each forcing, the run, each validation) is its own cell with a
**visual or numeric check** immediately after, so you can see the model take
shape piece by piece.

It is also written as a **reusable template for all of New Jersey**. Everything
that changes between regions/events lives in the single **`CONFIG`** cell below
— swap the region polygon, the time window, and the data-catalog entries and the
rest of the spine is unchanged.

**Pipeline:**
`Phase 1` static build (grid → elevation → mask → subgrid) →
`Phase 2` forcing (water level, wind, pressure, rain, discharge, infiltration; optional waves) →
`Phase 3` run SFINCS →
`Phase 4` results & validation (gauges, HWMs, FEMA MOTF extent).

## Modeling choices (the short version)

| Choice | What we do | Why |
|---|---|---|
| **Grid** | Rotated **quadtree**, 200 m base refined to 25 m at the surf zone | Resolves dunes/inlets where it matters, cheap offshore |
| **Subgrid** | 8×8 subgrid pixels per cell, V–h tables | Sub-cell topography on a coarse compute grid — SFINCS's core trick |
| **Elevation** | 5-layer merge, **pre-Sandy** topobathy on top | Post-storm DEMs bake in beach replenishment that didn't exist on 2012-10-29 |
| **Surge BC** | Observed **NOAA CO-OPS** water levels (not GTSM) | GTSM under-predicted Sandy by ~1 m at NJ latitudes |
| **Compound** | + ERA5 wind/pressure, AORC rain, USGS discharge, NRCS-CN infiltration | A true compound (surge + meteo + fluvial) flood |
| **Waves** | SnapWave incident + IG — **optional**, off by default | Research-grade; adds dune overtopping but is finicky (see Appendix) |

Datums are **NAVD88 metres** throughout. CRS is **UTM 18N (EPSG:32618)**.

## Setup

### Imports & environment

In [ ]:
import gc
import os
import subprocess
from datetime import datetime
from pathlib import Path

import geopandas as gpd
import hvplot.pandas  # noqa: F401  (registers .hvplot on GeoDataFrames)
import hvplot.xarray  # noqa: F401
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rioxarray  # noqa: F401  (registers .rio accessor)
import xarray as xr
from hydromt import log
from shapely.geometry import Point

from hydromt_sfincs import SfincsModel, utils

# Solver threads. SFINCS is OpenMP-parallel; set to the cores you have.
os.environ["OMP_NUM_THREADS"] = os.environ.get("OMP_NUM_THREADS", "48")


### Configuration — the one cell you edit per region/event

Everything region- or event-specific is collected here. To retarget this guide
to another NJ stretch or another storm, you should only need to touch `CONFIG`:
point `region` at a new polygon, set the `t*` dates, and make sure the data-
catalog entries cover the new footprint.

In [ ]:
CONFIG = {
    # ── Paths (relative to this notebook in notebooks/) ──────────────────────
    "model_root": "../model",  # build output dir (gitignored)
    "data_catalog": "../data/data_catalog.yml",  # single data library
    "region": "../data/region.geojson",  # model footprint polygon
    "refinement": "../data/quadtree/refinement_polygons.geojson",
    "obs": "../data/obs.geojson",  # observation points
    "container_sif": "../sfincs-desktop.sif",  # SFINCS Singularity image
    # ── Grid ────────────────────────────────────────────────────────────────
    "crs": "utm",  # let hydromt pick the UTM zone (→ 32618 here)
    "base_res": 200,  # level-0 cell size [m]; refinement steps down to ~25 m
    "rotated": True,  # rotate the grid to hug the coastline
    # ── Subgrid / mask ──────────────────────────────────────────────────────
    "nr_subgrid_pixels": 8,  # subgrid sampling per cell edge
    "mask_zmin": -10.0,  # cells with z >= this are active (NJ shelf)
    "reclass_table": "../data/roughness/NLCD_CONUS_mapping.csv",  # NJ-tuned Manning n
    # ── Elevation merge (top → bottom; first dataset with data wins) ─────────
    # See data_catalog.yml for the full provenance notes on each layer.
    "elevation_list": [
        {"elevation": "shrewsbury_ehydro_2015"},  # carve Rumson–Sea Bright bridge dam
        {"elevation": "usace_nj_2010"},  # 1 m PRE-Sandy topobathy (top)
        {"elevation": "cudem_nj"},  # 3 m fill: inlets + shelf + Raritan Bay
        {"elevation": "nj_10ft_dem", "zmin": 0.001},  # 3 m fill: inland land
        {"elevation": "gmrt_nj"},  # ~50 m GMRT offshore tail (was GEBCO)
    ],
    # ── Simulation window (Hurricane Sandy) ─────────────────────────────────
    "tref": datetime(2012, 10, 28),  # reference time
    "tstart": datetime(2012, 10, 28),  # ~24 h of calm tide before landfall
    "tstop": datetime(2012, 10, 31),
    "latitude": 40.32,  # for Coriolis (domain-mean lat)
    # ── Surge boundary (observed NOAA CO-OPS gauges) ────────────────────────
    "waterlevel_geodataset": "noaa_sandy_nj",
    "waterlevel_buffer": 100_000,  # m; reach down to Atlantic City gauge
    # ── Waves (SnapWave incident + IG) — OPTIONAL, see Appendix ──────────────
    "use_waves": False,  # flip True to add incident/IG wave forcing
    "wave_geodataset": "era5_waves_nj",
    "wave_era5_node": (-74.0, 40.0),  # nearest valid offshore ERA5 wave node
    "wave_n_support": 7,  # alongshore support points on the boundary
    "wavemaker_line": "../data/wavemakers/wavemaker_line.geojson",
    "dtwave": 1800.0,  # SnapWave coupling interval [s]
}

# A couple of derived convenience handles used throughout.
MODEL_ROOT = CONFIG["model_root"]
DATA_LIBS = [CONFIG["data_catalog"]]
print("model root :", MODEL_ROOT)
print("region     :", CONFIG["region"])
print("window     :", CONFIG["tstart"], "→", CONFIG["tstop"])
print("waves      :", "ON" if CONFIG["use_waves"] else "off (clean surge+meteo spine)")


---
## Phase 1 — Static build

Everything in Phase 1 (grid, elevation, mask, subgrid, obs points) is
**forcing-independent**. We build it once and write it to disk; Phase 2 reopens
from there. So while you iterate on water level / wind / rain you never rebuild
this expensive part.

### 1. Initialize the model

In [ ]:
log.initialize_logging()
log.set_log_level(log_level=30)  # warnings + errors only (quiet build)

Path(MODEL_ROOT).mkdir(parents=True, exist_ok=True)
log.to_file(Path(MODEL_ROOT) / "hydromt_sfincs.log", append=False)

# mode="w+" overwrites any previous build; write_gis dumps gis/*.tif|geojson
# for QGIS inspection as we go.
sf = SfincsModel(data_libs=DATA_LIBS, root=MODEL_ROOT, mode="w+", write_gis=True)
print("initialized empty SfincsModel at", MODEL_ROOT)


### 1b. Region — the model footprint

Before any grid, pin down the **footprint**. Everything downstream (grid,
elevation clips, mask, forcing) is bounded by this single polygon in
`CONFIG["region"]`, so it's worth looking at first.

For Sandy we deliberately reach beyond the open coast on both sides:

- **West** — an L-shaped lobe out to **−74.28** pulls in **Raritan Bay**, so
  surge and waves refract *through* the bay into Sandy Hook Bay and the back
  estuaries instead of hitting a wall at the bay mouth.
- **East** — the offshore edge runs out to **−73.45**, past the nearest **ERA5
  wave node (−73.5)**, so the wave-boundary support points sit in ~−35 m water
  *inside* the mesh (the X2 fix; see Appendix B).

> **Data coverage for this footprint** (audited 2026-06-29): CUDEM was re-clipped
> to cover **Raritan Bay** (3 m topobathy, ~−5 m mid-bay), and the offshore tail
> swapped GEBCO → **GMRT** (~50 m) to fill the new SE corner. So both new edges
> have real elevation data — see `data_catalog.yml`.


In [ ]:
# Region — the single polygon that bounds the whole build.
region = gpd.read_file(CONFIG["region"])
lon0, lat0, lon1, lat1 = region.total_bounds
reg_utm = region.to_crs(region.estimate_utm_crs())
print("Region footprint")
print(f"  lon/lat : [{lon0:.3f}, {lat0:.3f}] -> [{lon1:.3f}, {lat1:.3f}]")
print(f"  UTM 18N : {[round(v) for v in reg_utm.total_bounds]}")
print(f"  area    : {reg_utm.area.iloc[0] / 1e6:,.0f} km2")

# The offshore (east) edge is sized to reach the nearest ERA5 wave nodes (0.5 deg
# grid) so the wave-boundary input points sit inside the mesh, in real water.
era5_nodes = gpd.GeoDataFrame(
    geometry=[Point(-73.5, 40.0), Point(-73.5, 40.5)], crs="EPSG:4326"
)

# Zoomable check on satellite imagery: confirm Raritan Bay is captured on the
# west and the offshore edge clears the ERA5 nodes (green) on the east.
region.hvplot(
    geo=True, tiles="EsriImagery", fill_alpha=0.10, line_color="red",
    line_width=3, frame_width=760, frame_height=620,
    title="Model region — Raritan Bay (W) -> offshore ERA5 node (E)",
) * era5_nodes.hvplot.points(
    geo=True, color="lime", size=160, marker="^",
)


### 2. Build the quadtree grid

A **quadtree** keeps the domain coarse (200 m) offshore and recursively halves
the cell size toward the coast (→ 100 / 50 / 25 m) following the refinement
polygons. Passing `elevation_list` lets the level-2/3 polygons gate their
refinement by topobathy (e.g. only refine the dune/surf strip).

In [ ]:
refinement_gdf = gpd.read_file(CONFIG["refinement"])

sf.quadtree_grid.create_from_region(
    region={"geom": CONFIG["region"]},
    res=CONFIG["base_res"],
    rotated=CONFIG["rotated"],
    crs=CONFIG["crs"],
    refinement_polygons=refinement_gdf,
    elevation_list=CONFIG["elevation_list"],
)

# Cell-count breakdown. NOTE: hydromt_sfincs numbers levels 1-based, so
# level 1 IS the coarsest (= base_res); each step halves the cell size.
# The subgrid build (step 7) is the memory peak and scales with active cells ×
# nr_subgrid_pixels², so these per-level counts are the number to watch.
qg = sf.quadtree_grid.data
nlev = int(qg.attrs["nr_levels"])
n_total = int(qg.grid.n_face)
lev = qg["level"].values
print(f"Quadtree: {n_total:,} cells across {nlev} levels (level 1 = coarsest)")
for L in range(1, nlev + 1):
    n_cells = int((lev == L).sum())
    dx_lev = qg.attrs["dx"] / (2 ** (L - 1))
    print(f"  level {L}: {n_cells:>7,d} cells @ ~{dx_lev:.0f} m")


**Visual — the grid.** Cell centres coloured by refinement level. You should see the mesh tighten into a fine band along the open coast and inside the estuaries.

In [ ]:
fx, fy = sf.quadtree_grid.data.grid.face_coordinates.T
lev = sf.quadtree_grid.data["level"].values

fig, ax = plt.subplots(figsize=(7, 9))
sct = ax.scatter(fx, fy, c=lev, cmap="viridis", s=2, marker="s")
gpd.read_file(CONFIG["region"]).to_crs(sf.crs).boundary.plot(
    ax=ax, color="red", lw=1, label="region"
)
ax.set_aspect("equal")
ax.set_xlabel("Easting [m, UTM 18N]")
ax.set_ylabel("Northing [m]")
ax.set_title(f"Quadtree mesh — {len(fx):,} cells, {sf.crs.to_epsg()}")
fig.colorbar(sct, ax=ax, shrink=0.5, label="refinement level (1 = coarsest)")
plt.tight_layout()

### 3. Add elevation

The same 5-layer merge defined in `CONFIG["elevation_list"]` is sampled onto the
mesh — each refinement level at its own resolution (level 3 ≈ 25 m samples the
1 m USACE topobathy densely; level 0 ≈ 200 m samples GMRT). `nrmax` is the
per-level chunk size in **cells**: the merge tiles each level into `nrmax × nrmax`
blocks, reprojecting all five layers onto each block. Bigger blocks = fewer
blocks = less repeated read/reproject overhead (memory per block ∝ `nrmax²`), so
we use the library default `2000` rather than the old 24 GB-desktop `200`.

In [ ]:
sf.quadtree_elevation.create(
    elevation_list=CONFIG["elevation_list"], buffer_cells=0, nrmax=2000
)
z = sf.quadtree_grid.data["z"]
print(f"z range: {float(z.min()):.1f} .. {float(z.max()):.1f} m NAVD88")


**Visual — interactive topobathy (hvplot).** Mesh elevation rendered over Esri
imagery; pan/zoom to inspect the dune line, the inlets, and the dredged channels.
(Quadtree cells are rasterized on the fly, so this stays responsive.)

In [ ]:
# Rasterize the (rotated) quadtree mesh to a regular DEM, reproject to Web
# Mercator ONCE, then let datashader (rasterize=True) regrid it server-side.
# Two speed tricks vs the naive version:
#   - rasterize=True  -> datashader aggregates to screen resolution & only
#     re-renders the viewport, so a ~1 M-px raster stays responsive on zoom.
#   - reproject to 3857 + NO geo=True -> tiles already match the data CRS, so
#     cartopy doesn't re-project the whole image on every interaction.
_dem = sf.quadtree_grid.data["z"].ugrid.rasterize(resolution=50)  # 50 m, mesh CRS (UTM)
_dem = _dem.rio.write_crs(sf.crs).rio.reproject("EPSG:3857", nodata=float("nan"))
_dem.name = "z"

_dem.hvplot.image(
    x="x",
    y="y",
    rasterize=True,
    cmap="terrain",  # topobathy: blue water -> green/brown/white land
    clim=(-15, 15),
    tiles="EsriImagery",
    frame_width=650,
    frame_height=850,
    title="Topobathy on the quadtree mesh [m NAVD88]",
    clabel="bed level [m NAVD88]",
)


### 4. Active mask

Cells with `z ≥ CONFIG["mask_zmin"]` (-10 m) become **active** — the NJ shelf is
shallow enough that the -10 m contour is a good seaward edge. Everything deeper
is switched off.

In [ ]:
sf.quadtree_mask.create_active(zmin=CONFIG["mask_zmin"])
m = sf.quadtree_grid.data["mask"].values
print(f"active cells: {int((m > 0).sum()):,} / {sf.quadtree_grid.data.grid.n_face:,}")

### 5. Boundary cells

Standard rule first: **waterlevel** boundary on the deep (`z ≤ -1`) perimeter
edges, **outflow** on the shallow lateral edges. Then two *region-specific*
coordinate boxes fix estuary mis-classifications without tracing the thin Sea
Bright barrier:

- **(a)** western estuary edges below the bay (Navesink @ Red Bank, mainland) →
  **outflow**: they drain from inside the domain, they aren't driven by the ocean.
- **(b)** a small Shrewsbury back-channel pocket west of the seaward edge →
  plain **active** interior.

> ⚠️ **These boxes are Sandy-specific UTM coordinates.** For a different NJ
> region, re-derive them (or drop them and inspect the default mask first).
> The Sea Bright→Highlands seaward strip deliberately *stays* waterlevel — east
> of it the shelf is deep/inactive, so that strip **is** the open-ocean boundary.

In [ ]:
sf.quadtree_mask.create_boundary(btype="waterlevel", zmax=-1, reset_bounds=True)
sf.quadtree_mask.create_boundary(btype="outflow", zmin=-1, zmax=2, reset_bounds=False)

# --- region-specific geographic corrections (NJ-Sandy; UTM 18N) ---
mask = sf.quadtree_grid.data["mask"].values.copy()
fx, fy = sf.quadtree_grid.data.grid.face_coordinates.T

west_below_bay = (fx < 582_500) & (fy < 4_474_000)                       # (a)
shrewsbury = (fx > 586_500) & (fx < 587_400) & (fy > 4_467_000) & (fy < 4_472_000)  # (b)
mask[(mask == 2) & west_below_bay] = 3   # waterlevel → outflow
mask[(mask == 2) & shrewsbury] = 1       # waterlevel → active interior
sf.quadtree_grid.data["mask"] = sf.quadtree_grid.data["mask"].copy(data=mask)

m = sf.quadtree_grid.data["mask"].values
print(f"active   (1): {int((m == 1).sum()):,}")
print(f"waterlevel(2): {int((m == 2).sum()):,}")
print(f"outflow  (3): {int((m == 3).sum()):,}")

**Visual — the mask.** Inactive cells dropped; active interior grey, waterlevel (driven) boundary in blue, outflow in orange.

In [ ]:
mvals = sf.quadtree_grid.data["mask"].values
cat_color = {1: ("0.6", "active"), 2: ("tab:blue", "waterlevel"), 3: ("tab:orange", "outflow")}
fig, ax = plt.subplots(figsize=(7, 9))
for val, (col, lab) in cat_color.items():
    sel = mvals == val
    ax.scatter(fx[sel], fy[sel], c=col, s=3, marker="s", label=f"{lab} ({int(sel.sum()):,})")
ax.set_aspect("equal")
ax.set_xlabel("Easting [m]"); ax.set_ylabel("Northing [m]")
ax.set_title("SFINCS mask")
ax.legend(loc="upper left", markerscale=3)
plt.tight_layout()

### 6. Observation points

Project the obs points plus the in-domain **Sandy Hook NOAA gauge (8531680)** —
modeled `zs(t)` here vs the observed record is our cleanest single-point check
(the gauge failed mid-storm ~23:00 UTC 10-29, so it's valid up to then).

In [ ]:
obs_gdf = gpd.read_file(CONFIG["obs"])
sandy_hook = gpd.GeoDataFrame(
    {"name": ["sandy_hook_gauge"]},
    geometry=[Point(-74.0091, 40.4669)],  # NOAA 8531680
    crs="EPSG:4326",
)
obs_all = gpd.GeoDataFrame(
    pd.concat([obs_gdf, sandy_hook], ignore_index=True), crs="EPSG:4326"
)
sf.observation_points.create(locations=obs_all, merge=False)

_ = sf.plot_basemap(variable="dep", plot_geoms=True, plot_bounds=True, bmap="sat", zoomlevel=12)

### 7. Roughness + subgrid tables

This is the memory/CPU peak. `quadtree_roughness.create` writes a per-cell
Manning *n* from the NJ-tuned NLCD reclass; `quadtree_subgrid.create` then
samples each cell at `nr_subgrid_pixels`² finer points to build the
volume / conveyance / roughness lookup tables that give SFINCS sub-cell
topography on a coarse compute grid.

`nrmax=2000` for the subgrid is **load-bearing** — smaller values explode the
number of blocks and the inner Python loop (minutes → hours). Don't lower it.

In [ ]:
# Drop the elevation-step DEM cache before subgrid reloads them, or peak memory
# doubles (hydromt keeps every RasterDataset it read alive on the catalog).
for src in list(sf.data_catalog.sources):
    s = sf.data_catalog.get_source(src)
    if hasattr(s, "_data"):
        s._data = None
gc.collect()

roughness_list = [{"lulc": "nlcd_2012", "reclass_table": CONFIG["reclass_table"]}]
sf.quadtree_roughness.create(roughness_list=roughness_list, nrmax=200)

sf.quadtree_subgrid.create(
    elevation_list=CONFIG["elevation_list"],
    roughness_list=roughness_list,
    nr_subgrid_pixels=CONFIG["nr_subgrid_pixels"],
    nrmax=2000,            # DO NOT lower — see markdown above
    write_dep_tif=True,    # per-level subgrid DEMs (used by the flood-map downscale)
    write_man_tif=True,
)

**Check — subgrid table ranges.** On quadtree the subgrid lives at `sf.quadtree_subgrid.data` with per-edge variables; just sanity-check the ranges.

In [ ]:
sg = sf.quadtree_subgrid.data
print("subgrid variables:", list(sg.data_vars))
for v in ["z_zmin", "z_zmax", "uv_havg", "uv_navg"]:
    if v in sg:
        print(f"  {v}: shape={tuple(sg[v].shape)}  min={float(sg[v].min()):.3f}  max={float(sg[v].max()):.3f}")

### 8. Write the static model

In [ ]:
sf.write()
print("static model written to", MODEL_ROOT)

# Free build-time memory; Phase 2 opens a fresh handle and needs none of it.
del sf
gc.collect()

---
## Phase 2 — Forcing

Reopen the static model and layer on the forcings. With `CONFIG["use_waves"]`
off, this is a clean **compound surge + meteo + fluvial** model: observed surge
boundary, ERA5 wind & pressure, AORC rain, USGS discharge, and NRCS-CN
infiltration.

### Reopen the static model

In [ ]:
sf = SfincsModel(MODEL_ROOT, data_libs=DATA_LIBS, mode="r+")
print(f"reopened {sf.grid_type} model at {MODEL_ROOT}")

### 1. Simulation window + water-level boundary

Set the run period and physics flags, then apply the observed **NOAA CO-OPS**
water levels to the boundary cells. `merge=False` replaces any stale boundary
from a previous build; the 100 km buffer reaches the Atlantic City gauge.

In [ ]:
sf.config.update(
    {
        "tref": CONFIG["tref"],
        "tstart": CONFIG["tstart"],
        "tstop": CONFIG["tstop"],
        "tspinup": 3600.0,
        "coriolis": 1,
        "latitude": CONFIG["latitude"],
        "advection": 1,
        "dtmapout": 3600.0,    # map output every hour
        "dtmaxout": 86400.0,   # one zsmax over the whole run
        "dthisout": 600.0,     # his (obs-point) output every 10 min
    }
)

sf.water_level.create(
    geodataset=CONFIG["waterlevel_geodataset"],
    buffer=CONFIG["waterlevel_buffer"],
    merge=False,
)
print("water-level boundary points:", sf.forcing["bzs"].sizes if "bzs" in sf.forcing else "(see sf.forcing)")

### 2. Wind + pressure (ERA5)

In [ ]:
# ERA5 file is pre-renamed to hydromt conventions (wind10_u/v m/s, press_msl Pa).
sf.wind.create(wind="era5_nj")
sf.pressure.create(press="era5_nj")
print("added ERA5 wind + pressure")

### 3. Rainfall (NOAA AORC)

In [ ]:
# AORC is accumulated mm per 1-h interval → cumulative_input=True.
# aggregate=False keeps it spatially distributed.
sf.precipitation.create(precip="aorc_sandy_nj", cumulative_input=True, aggregate=False)
_pr = sf.precipitation.data
_var = "precip_2d" if "precip_2d" in _pr else list(_pr.data_vars)[0]
print(f"precip [mm/hr]: peak={float(_pr[_var].max()):.2f}  mean={float(_pr[_var].mean()):.3f}")

### 4. River discharge (USGS)

In [ ]:
# Daily-mean discharge at the 2 domain inflows (Shark River, Navesink).
sf.discharge_points.create(geodataset="usgs_sandy_discharge", merge=False)
_dis = sf.discharge_points.data
print(f"discharge points: {_dis.sizes.get('index', _dis.sizes.get('stations'))}  "
      f"peak={float(_dis['dis'].max()):.2f} m3/s")

### 5. Infiltration (NRCS Curve Number)

SCS-CN infiltration on the mesh (`antecedent_moisture=None` → CN II / average).
Surge-dominated Sandy means this barely moves validation (±0.02 CSI), but it's
part of a proper compound build.

> 🐛 **Known hydromt-sfincs v2.0.0rc2 bug:** the quadtree infiltration component
> sets the `infiltration_file` key in `sfincs.inp` but its `write()` is a no-op,
> so no file is emitted and SFINCS aborts. The finalize cell strips those orphan
> keys. (Tracked; remove the patch once upstream is fixed.)

In [ ]:
sf.quadtree_infiltration.create_cn(cn="cn_nj", antecedent_moisture=None, nrmax=2000)
_scs = sf.quadtree_grid.data["scs"]
print(f"SCS retention S [inch]: mean={float(_scs.where(_scs > 0).mean()):.2f}  "
      f"max={float(_scs.max()):.2f}  (S=0 over water/impervious)")

### 6. *(Optional)* Incident & IG waves — SnapWave

Runs only if `CONFIG["use_waves"]` is `True`. Injects the ERA5 incident-wave
spectrum (uniform alongshore) at the offshore waterlevel boundary plus an
IG-wave wavemaker line. This is research-grade and finicky — read the
**Appendix** before turning it on. With waves off, this cell is a no-op.

In [ ]:
snapwave_pts = snapwave_t = snapwave_hs = snapwave_tp = snapwave_wd = snapwave_ds = None

if CONFIG["use_waves"]:
    N = CONFIG["wave_n_support"]
    _grid = sf.quadtree_grid.data
    _bxy = _grid.grid.face_coordinates[_grid["mask"].values == 2]  # seaward boundary cells
    _ybins = np.linspace(_bxy[:, 1].min(), _bxy[:, 1].max(), N + 1)
    # support points = easternmost (seaward) boundary cell per northing bin
    snapwave_pts = np.array([
        grp[np.argmax(grp[:, 0])]
        for k in range(N)
        for grp in [_bxy[(_bxy[:, 1] >= _ybins[k]) & (_bxy[:, 1] <= _ybins[k + 1])]]
        if len(grp)
    ])

    _ew = sf.data_catalog.get_rasterdataset(CONFIG["wave_geodataset"])
    _node = _ew.sel(x=CONFIG["wave_era5_node"][0], y=CONFIG["wave_era5_node"][1], method="nearest")
    snapwave_t = (_node["time"].values - _node["time"].values[0]) / np.timedelta64(1, "s")
    snapwave_hs, snapwave_tp, snapwave_wd = _node["hs"].values, _node["tp"].values, _node["wd"].values
    snapwave_ds = np.full_like(snapwave_hs, 30.0)  # ERA5 has no dir-spreading; 30 deg

    sf.config.update({"snapwave": 1, "snapwave_igwaves": 1, "dtwave": CONFIG["dtwave"]})
    sf.wave_makers.create(CONFIG["wavemaker_line"], merge=False)
    print(f"waves ON: {len(snapwave_pts)} support points, "
          f"hs {snapwave_hs.min():.1f}-{snapwave_hs.max():.1f} m, wavemaker line added")
else:
    print("waves off — clean surge+meteo spine")

### 7. Write the model + finalize `sfincs.inp`

`sf.write()` emits everything, then we patch `sfincs.inp` on disk for two known
hydromt-sfincs v2.0.0rc2 issues (and, if waves are on, write the SnapWave ASCII
forcing). Every patch is idempotent — delete it once upstream is fixed.

In [ ]:
sf.write()

root = Path(MODEL_ROOT)
inp = root / "sfincs.inp"
text = inp.read_text()

# (a) latitude — dropped on write, so Coriolis silently disables without it.
if "\nlatitude" not in text:
    text = text.replace(
        "coriolis             = 1",
        f"coriolis             = 1\nlatitude             = {CONFIG['latitude']}",
    )
    print(f"patched latitude = {CONFIG['latitude']}")

# (b) strip orphan infiltration keys (component sets key but writes no file).
text = "\n".join(
    ln for ln in text.splitlines()
    if not ln.strip().startswith(("infiltration_file", "infiltration_type", "scsfile"))
) + "\n"

# (c) waves: ensure SnapWave keys + write the ASCII boundary forcing.
if CONFIG["use_waves"]:
    sw_keys = {
        "snapwave": "1", "snapwave_igwaves": "1", "dtwave": str(CONFIG["dtwave"]),
        "wvmfile": "sfincs.wvm",
        "snapwave_bndfile": "snapwave.bnd", "snapwave_bhsfile": "snapwave.bhs",
        "snapwave_btpfile": "snapwave.btp", "snapwave_bwdfile": "snapwave.bwd",
        "snapwave_bdsfile": "snapwave.bds",
    }
    present = {ln.split("=")[0].strip() for ln in text.splitlines() if "=" in ln}
    for k, v in sw_keys.items():
        if k not in present:
            text += f"{k:<20} = {v}\n"
    for stale in ("snapwave.upw", "snapwave.nc"):  # keyed to old config → crash if stale
        (root / stale).unlink(missing_ok=True)
    np.savetxt(root / "snapwave.bnd", snapwave_pts, fmt="%.3f")
    for fn, series in [("snapwave.bhs", snapwave_hs), ("snapwave.btp", snapwave_tp),
                       ("snapwave.bwd", snapwave_wd), ("snapwave.bds", snapwave_ds)]:
        block = np.tile(np.asarray(series)[:, None], (1, len(snapwave_pts)))
        np.savetxt(root / fn, np.column_stack([snapwave_t, block]),
                   fmt=["%11.1f"] + ["%11.3f"] * len(snapwave_pts))
    print("wrote SnapWave keys + snapwave.{bnd,bhs,btp,bwd,bds}")

inp.write_text(text)
print("finalized", inp)

---
## Phase 3 — Run SFINCS

The solver runs in the official `deltares/sfincs-cpu` container. The helper
auto-detects **Singularity** (HPC/Amarel — no Docker daemon allowed) or
**Docker** (local desktop) so the same cell runs everywhere.

> For long/production solves on Amarel, prefer `sbatch hpc/sfincs_run.slurm <model_dir>`
> on a compute node — never solve on the login node.

In [ ]:
def run_sfincs(model_root, sif="../sfincs-cpu.sif"):
    """Run SFINCS in the deltares/sfincs-cpu container (Singularity or Docker)."""
    import shutil
    model_abs = Path(model_root).resolve()
    log_path = model_abs / "sfincs_log.txt"
    threads = os.environ.get("OMP_NUM_THREADS") or str(os.cpu_count() or 1)

    # Clear stale outputs first — a held-open sfincs_map.nc/his.nc triggers HDF5
    # file-locking that makes SFINCS silently write ZERO output.
    for stale in ("sfincs_map.nc", "sfincs_his.nc"):
        try:
            (model_abs / stale).unlink()
        except FileNotFoundError:
            pass

    if shutil.which("singularity"):
        sif_abs = Path(os.environ.get("SFINCS_SIF", sif)).resolve()
        print(f"Running SFINCS via Singularity ({sif_abs.name}) [OMP={threads}] ...")
        env = {**os.environ, "OMP_NUM_THREADS": threads, "SINGULARITYENV_OMP_NUM_THREADS": threads}
        with open(log_path, "w") as lf:
            return subprocess.run(
                ["singularity", "run", "--bind", f"{model_abs}:/data", "--pwd", "/data", str(sif_abs)],
                stdout=lf, stderr=subprocess.STDOUT, env=env,
            )
    if shutil.which("docker"):
        print(f"Running SFINCS via Docker [OMP={threads}] ...")
        subprocess.run(  # clear root-owned stale outputs from a prior Docker run
            ["docker", "run", "--rm", "-v", f"{model_abs}:/data", "--entrypoint", "/bin/sh",
             "deltares/sfincs-cpu:latest", "-c", "rm -f /data/sfincs_map.nc /data/sfincs_his.nc"],
            capture_output=True,
        )
        with open(log_path, "w") as lf:
            return subprocess.run(
                ["docker", "run", "--rm", "-v", f"{model_abs}:/data", "deltares/sfincs-cpu:latest"],
                stdout=lf, stderr=subprocess.STDOUT,
            )
    raise RuntimeError("Neither 'singularity' nor 'docker' on PATH.")


os.environ.setdefault("SFINCS_SIF", str(Path(CONFIG["container_sif"]).resolve()))
result = run_sfincs(MODEL_ROOT, sif=CONFIG["container_sif"])
print(f"Done (return code {result.returncode})")

**Check — log tail + output files.**

In [ ]:
log_path = Path(MODEL_ROOT) / "sfincs_log.txt"
if log_path.exists():
    print("\n".join(log_path.read_text().splitlines()[-25:]))
print("\nsfincs_map.nc:", (Path(MODEL_ROOT) / "sfincs_map.nc").exists(),
      " sfincs_his.nc:", (Path(MODEL_ROOT) / "sfincs_his.nc").exists())

---
## Phase 4 — Results & validation

Open the run read-only and ask the central question: **did we capture Sandy?**
We check (1) the Sandy Hook gauge time series, (2) the downscaled flood map,
(3) USGS High Water Marks (spatial over/under), and (4) the FEMA MOTF extent
(CSI/POD/FAR).

### Open the results

In [ ]:
mod = SfincsModel(MODEL_ROOT, data_libs=DATA_LIBS, mode="r")
mod.output.read()
model_abs = Path(MODEL_ROOT).resolve()
print("grid_type:", mod.grid_type)
print("output vars:", list(mod.output.data.keys()))

### Model layout

In [ ]:
fig, ax = mod.plot_basemap(fn_out=None, bmap="sat", figsize=(9, 7), geom_names=["obs"])

### Validation 1 — Sandy Hook gauge (temporal)

In [ ]:
point_zs = mod.output.data["point_zs"]   # (time, station)
point_zb = mod.output.data["point_zb"]   # (station,)
names = [n.decode() if isinstance(n, bytes) else str(n) for n in point_zs["station_name"].values]

val = xr.open_dataset("../data/gtsm/noaa_sandy_validation.nc")
obs_sh = val["waterlevel"].sel(stations=8531680)

i_sh = next(k for k, n in enumerate(names) if "sandy_hook" in n)
mod_sh = point_zs.isel(stations=i_sh)
zb_sh = float(point_zb.isel(stations=i_sh).values)
mod_sh_wet = mod_sh.where(mod_sh - zb_sh > 0.01)

gauge_end = pd.Timestamp("2012-10-29 23:00")
print(f"observed peak (pre-failure): {float(obs_sh.max()):.2f} m NAVD88")
print(f"modeled  peak (same window): {float(mod_sh.sel(time=slice(None, gauge_end)).max()):.2f} m")
print(f"modeled  peak (full run):    {float(mod_sh.max()):.2f} m")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(mod_sh["time"], mod_sh_wet.values, lw=2, label="modeled zs (SFINCS)")
ax.plot(obs_sh["time"], obs_sh.values, "k.-", ms=4, label="observed (NOAA 8531680)")
ax.axvline(gauge_end, color="red", ls=":", alpha=0.6, label="gauge fails 10-29 23:00")
ax.set_ylabel("WSE [m NAVD88]"); ax.set_xlabel("Time [UTC]")
ax.set_title("Sandy Hook: modeled vs observed water level")
ax.legend(); ax.grid(alpha=0.3); fig.autofmt_xdate(); plt.tight_layout()

### Downscale the flood map

`zsmax` is downscaled onto the finest subgrid DEM (L3, ~3 m) for a crisp flood
extent. On a rotated grid, passing the DEM as a *path* + `floodmap_fn` makes
hydromt tile the downscale (bounded memory) and write to disk; we read it back
and de-rotate to a north-up grid so later row/col sampling is correct.

In [ ]:
da_zsmax = mod.output.data["zsmax"].max(dim="timemax")
depfile = str(model_abs / "subgrid" / "dep_subgrid_lev3.tif")
floodmap_fn = str(model_abs / "floodmap_hmax_lev3.tif")

utils.downscale_floodmap(zsmax=da_zsmax, dep=depfile, hmin=0.05, floodmap_fn=floodmap_fn, nrmax=1000)

da_hmax = rioxarray.open_rasterio(floodmap_fn, masked=True).squeeze(drop=True)
da_dep = rioxarray.open_rasterio(depfile, masked=True).squeeze(drop=True)
da_hmax = da_hmax.rio.reproject(da_hmax.rio.crs)        # de-rotate to north-up
da_dep = da_dep.rio.reproject_match(da_hmax)            # onto identical grid
da_hmax = da_hmax.where(da_dep.values > -0.5)           # drop deep ocean
da_hmax.name = "hmax"
print(f"flood map (L3): {tuple(da_hmax.shape)}, res {da_hmax.rio.resolution()[0]:.1f} m")

**Flood map — max water depth.**

In [ ]:
fig, ax = mod.plot_basemap(
    fn_out=None, figsize=(8, 6), variable=da_hmax, plot_bounds=False, plot_geoms=False,
    bmap="sat", zoomlevel=11, vmin=0, vmax=5.0, cbar_kwargs={"shrink": 0.6, "anchor": (0, 0)},
)
ax.set_title("SFINCS maximum water depth [m]")

**Interactive flood map (hvplot).** Max depth over Esri imagery; obs points in red.

In [ ]:
obs_all = gpd.read_file(model_abs / "gis" / "obs.geojson")
flood_wgs84 = da_hmax.where(da_hmax > 0.05).rio.reproject("EPSG:4326", nodata=float("nan"))
flood_map = flood_wgs84.hvplot.image(
    x="x", y="y", cmap="viridis", clim=(0, 5), geo=True, tiles="EsriImagery",
    alpha=0.75, frame_width=850, frame_height=650,
    title="Max flood depth — Hurricane Sandy", clabel="Flood depth [m]",
)
obs_layer = obs_all.hvplot.points(geo=True, color="red", size=60, hover_cols=["name"])
flood_map * obs_layer

### Validation 2 — USGS High Water Marks (spatial)

For each USGS Sandy HWM, find the modeled peak still-water surface within 50 m
(only genuinely flooded cells, and only cells whose *ground* sits at/below the
mark, so we don't grab a higher dune). Residual = model − observed.

In [ ]:
DEPTH_MIN, GROUND_CAP = 0.15, 0.5
hwm = gpd.read_file("../data/validation/sandy_hwms.geojson").to_crs(da_dep.rio.crs)

depth, dep_arr, wse = da_hmax.values, da_dep.values, (da_dep + da_hmax).values
if depth.ndim == 3:
    depth, wse, dep_arr = depth[0], wse[0], dep_arr[0]
T = da_dep.rio.transform()
ny, nx = wse.shape
rad = int(round(50 / abs(T.a)))

obs = hwm["elev_m"].values
qual = hwm["quality"].values.astype(float)
mod_wse = np.full(len(obs), np.nan)
for k, (X, Y) in enumerate(zip(hwm.geometry.x.values, hwm.geometry.y.values)):
    col, row = int((X - T.c) / T.a), int((Y - T.f) / T.e)
    if 0 <= row < ny and 0 <= col < nx:
        sl = (slice(max(0, row - rad), row + rad + 1), slice(max(0, col - rad), col + rad + 1))
        ws, hh, dd = wse[sl], depth[sl], dep_arr[sl]
        flooded = (hh >= DEPTH_MIN) & (dd <= obs[k] + GROUND_CAP)
        if flooded.any():
            mod_wse[k] = np.nanmax(np.where(flooded, ws, np.nan))

wet = np.isfinite(mod_wse)
resid = mod_wse - obs
q2 = qual <= 2

def report(label, msk):
    if msk.sum() == 0:
        print(f"{label}: (none)"); return
    r = resid[msk]
    print(f"{label}: n={int(msk.sum()):2d}  mean={r.mean():+.2f}  median={np.median(r):+.2f}  "
          f"RMSE={np.sqrt((r**2).mean()):.2f}  within±0.5m={np.mean(np.abs(r)<0.5)*100:.0f}%")

print(f"model flooded within 50 m of {wet.sum()}/{len(obs)} HWMs  (+ = over-predict)")
report("HEADLINE q<=2", wet & q2)
report("         q<=3", wet & (qual <= 3))
report("         all ", wet)
print(f"model DRY at {int((~wet).sum())} HWMs (still-water can't reach — runup candidates)")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(obs[wet & ~q2], mod_wse[wet & ~q2], facecolor="none", edgecolor="grey",
           s=55, lw=0.8, label="q3-4")
sc = ax.scatter(obs[wet & q2], mod_wse[wet & q2], c=qual[wet & q2], cmap="viridis_r",
                s=60, edgecolor="k", lw=0.4, vmin=1, vmax=5, label="q1-2 (headline)")
lim = [1.8, 6.0]
ax.plot(lim, lim, "k--", lw=1, label="1:1")
ax.fill_between(lim, [l - 0.5 for l in lim], [l + 0.5 for l in lim], color="grey", alpha=0.15)
ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect("equal")
ax.set_xlabel("Observed HWM [m NAVD88]"); ax.set_ylabel("Modeled still-water WSE [m NAVD88]")
ax.set_title("Modeled still-water vs USGS HWMs")
fig.colorbar(sc, ax=ax, shrink=0.8, label="HWM quality (1=best)")
ax.legend(loc="upper left"); ax.grid(alpha=0.3); plt.tight_layout()

**HWM residual map** — red = model over-predicts, blue = under, ✕ = model dry.

In [ ]:
fig, ax = mod.plot_basemap(
    fn_out=None, figsize=(9, 7), variable=da_hmax, plot_bounds=False, plot_geoms=False,
    bmap="sat", zoomlevel=11, vmin=0, vmax=5, cmap="Blues",
    cbar_kwargs={"shrink": 0.5, "label": "Modeled depth [m]"},
)
hx, hy = hwm.geometry.x.values, hwm.geometry.y.values
sc = ax.scatter(hx[wet], hy[wet], c=resid[wet], cmap="RdBu_r", vmin=-1.5, vmax=1.5,
                s=70, edgecolor="k", lw=0.6, zorder=5)
ax.scatter(hx[~wet], hy[~wet], marker="x", color="k", s=70, lw=1.6, zorder=6,
           label=f"model dry ({int((~wet).sum())})")
fig.colorbar(sc, ax=ax, shrink=0.5, label="HWM residual: model − obs [m]")
ax.legend(loc="upper right"); ax.set_title("Sandy HWM residuals"); plt.tight_layout()

### Validation 3 — FEMA MOTF extent (CSI / POD / FAR)

Both rasters are EPSG:32618, so we sample model cells at MOTF pixel centres in
pure numpy (no GDAL warp). CSI is the headline skill score; the categorical map
shows hits / misses / false-alarms.

In [ ]:
import rasterio
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

DEPTH_MIN = 0.15
with rasterio.open("../data/validation/sandy_motf_extent.tif") as r:
    motf, mtf, m_nd = r.read(1), r.transform, r.nodata
mod_t = da_dep.rio.transform()
mh, mw = motf.shape

Xc = mtf.c + (np.arange(mw) + 0.5) * mtf.a
Yc = mtf.f + (np.arange(mh) + 0.5) * mtf.e
mc = np.clip(((Xc - mod_t.c) / mod_t.a).astype(int), 0, da_dep.shape[-1] - 1)
mr = np.clip(((Yc - mod_t.f) / mod_t.e).astype(int), 0, da_dep.shape[-2] - 1)
rr, cc = np.meshgrid(mr, mc, indexing="ij")
_2d = lambda a: a[0] if a.ndim == 3 else a
dep_at, h_at = _2d(da_dep.values)[rr, cc], _2d(da_hmax.values)[rr, cc]

motf_wet = motf == 1
mod_wet = (h_at >= DEPTH_MIN) & np.isfinite(h_at)
land_in = (motf != m_nd) & (dep_at > 0.0)
hits, miss, fa = motf_wet & mod_wet & land_in, motf_wet & ~mod_wet & land_in, ~motf_wet & mod_wet & land_in
nh, nm, nf = int(hits.sum()), int(miss.sum()), int(fa.sum())
PIX = mtf.a * abs(mtf.e) / 1e6
CSI = nh / (nh + nm + nf)
POD = nh / (nh + nm) if (nh + nm) else 0.0
FAR = nf / (nh + nf) if (nh + nf) else 0.0
print(f"MOTF flooded land : {(motf_wet & land_in).sum() * PIX:.1f} km2")
print(f"Model wet land    : {(mod_wet & land_in).sum() * PIX:.1f} km2")
print(f"  hits {nh*PIX:5.1f}  miss {nm*PIX:5.1f}  false-alarm {nf*PIX:5.1f} km2")
print(f"  CSI={CSI:.2f}  POD={POD:.2f}  FAR={FAR:.2f}")

cat = np.zeros_like(motf, dtype="uint8")
cat[hits], cat[miss], cat[fa] = 1, 2, 3
cmap = ListedColormap([(1,1,1,0), (.2,.6,.3,1), (.2,.4,.85,1), (.85,.2,.2,1)])
ext = [mtf.c, mtf.c + mw*mtf.a, mtf.f + mh*mtf.e, mtf.f]
mod_ext = [mod_t.c, mod_t.c + da_dep.shape[-1]*mod_t.a, mod_t.f + da_dep.shape[-2]*mod_t.e, mod_t.f]

fig, ax = plt.subplots(figsize=(7.5, 9))
ax.imshow(_2d(da_dep.values), extent=mod_ext, cmap="Greys", vmin=-5, vmax=20, alpha=0.45, origin="upper")
ax.imshow(cat, cmap=cmap, vmin=0, vmax=3, extent=ext, origin="upper", interpolation="nearest")
ax.set_aspect("equal"); ax.set_xlim(ext[0], ext[1]); ax.set_ylim(ext[2], ext[3])
ax.set_xlabel("Easting [m]"); ax.set_ylabel("Northing [m]")
ax.legend(handles=[
    Patch(color=cmap(1), label=f"hit ({nh*PIX:.1f} km²)"),
    Patch(color=cmap(2), label=f"miss ({nm*PIX:.1f} km²)"),
    Patch(color=cmap(3), label=f"false alarm ({nf*PIX:.1f} km²)"),
], loc="upper right", fontsize=8)
ax.set_title(f"Modeled flood vs FEMA MOTF — CSI={CSI:.2f}  POD={POD:.2f}  FAR={FAR:.2f}")
plt.tight_layout()

### Diagnostic — bridges baked into the bed as dams

Interactive bed map of the Navesink/Shrewsbury crossings. Channels read blue
(<0); an earthen bridge causeway reads red (>0) cutting across — exactly the
artifact the eHydro channel-survey layer (top of the elevation merge) carves
back open so tide + surge can propagate up-estuary.

In [ ]:
from rasterio.enums import Resampling

_bed = rioxarray.open_rasterio(model_abs / "subgrid" / "dep_subgrid_lev3.tif", masked=True).squeeze(drop=True)
_bed = _bed.rio.reproject(_bed.rio.crs, resolution=3.0, resampling=Resampling.min)  # keep deepest sub-pixel
_bed = _bed.rio.clip_box(581_000, 4_463_000, 588_500, 4_476_000).rio.reproject("EPSG:4326")
_bed.name = "bed_m"
_bed.hvplot.image(
    x="x", y="y", cmap="RdBu_r", clim=(-6, 6), geo=True, tiles="EsriImagery",
    alpha=0.8, frame_width=700, frame_height=900, clabel="bed [m NAVD88]",
    title="Model bed — channel (blue) vs bridge dam (red). Zoom to the crossings.",
)

---
## Appendix A — Extending to all of New Jersey

This notebook is built to generalize. To move beyond the Sandy Hook → Asbury
footprint:

1. **Region** — swap `CONFIG["region"]` for the new polygon (and rebuild the
   `refinement_polygons` for the new coast).
2. **Data catalog** — the rasters in `data/data_catalog.yml` are clipped to the
   current bbox. Re-clip/re-download (the `scripts/download_*.py` are
   parameterized by region) so each layer covers the new footprint.
3. **Boundary boxes** — the two coordinate boxes in Phase 1 step 5 are
   Sandy-specific. Re-derive them for the new estuaries, or start from the
   default mask and inspect before adding corrections.
4. **Event window + forcing** — set `CONFIG["t*"]` and make sure the NOAA gauges
   / ERA5 / AORC / USGS series cover the new dates and area.

Keep the *spine* (the build/forcing/validation cells) untouched — that's the
point of the config-driven design.

## Appendix B — On the optional wave forcing (`use_waves`)

SnapWave incident + IG waves add dune-overtopping runup the still-water model
omits, but the setup is finicky and **not yet a validated net-positive** for this
domain. Known pitfalls (all handled in the gated cells when `use_waves=True`):

- **Input points must lie inside the mesh, in real water.** We place support
  points on the seaward `mask==2` boundary (not at raw ERA5 coords, which sit
  outside the mesh → depth 0 → IG blow-up → SFINCS crash).
- The forcing is **uniform alongshore** — the ERA5 0.5° wave grid (~55 km) is
  too coarse to resolve ~40 km of coast; only one offshore node is valid.
- Stale `snapwave.upw` / `snapwave.nc` from a previous config will crash the
  solver; the finalize cell deletes them.

Turn it on only once the clean spine validates, and expect to iterate.

## Status & open questions

- **Back-bay conveyance** is the main residual: surge reaches the bay mouth but
  under-propagates up the Navesink/Shrewsbury to Oceanport. The eHydro bridge
  carve helped; narrows resolution is the next lever.
- **Open-coast extent** is captured well; false alarms are minor (Sandy Hook
  spit + a few southern spots).
- **Waves / IG runup** is the candidate for the HWMs the still-water model
  leaves dry — see Appendix B.